In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, 
    precision_recall_curve, roc_curve, f1_score, 
    precision_score, recall_score, accuracy_score
)

import xgboost as xgb
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

print("Libraries imported successfully!")
print(f"Python version: {pd.__version__}")

In [ ]:
data_path = r"c:\Users\siddhi\Desktop\Projects-20240722T093004Z-001\Projects\fraud_detection\fraud_detection\dataset\data"

pickle_files = [f for f in os.listdir(data_path) if f.endswith('.pkl')]
pickle_files.sort()  

print(f"Data path: {data_path}")
print(f"Number of daily files: {len(pickle_files)}")
print(f"Date range: {pickle_files[0]} to {pickle_files[-1]}")
print(f"\nFirst 10 files: {pickle_files[:10]}")
print(f"Last 10 files: {pickle_files[-10:]}")

In [ ]:
def load_daily_data(file_path):
    """Load a single pickle file and return the dataframe"""
    with open(file_path, 'rb') as f:
        data = pickle.load(f)
    return data

sample_file = os.path.join(data_path, pickle_files[0])
sample_data = load_daily_data(sample_file)

print(f"Sample file: {pickle_files[0]}")
print(f"Data type: {type(sample_data)}")
print(f"Shape: {sample_data.shape}")
print(f"\nColumns: {list(sample_data.columns)}")
print(f"\nFirst few rows:")
sample_data.head()

In [ ]:
print("Data Info:")
print(sample_data.info())
print("\n" + "="*50)
print("Data Description:")
print(sample_data.describe())
print("\n" + "="*50)
print("Missing Values:")
print(sample_data.isnull().sum())
print("\n" + "="*50)
print("Unique Values per Column:")
for col in sample_data.columns:
    print(f"{col}: {sample_data[col].nunique()} unique values")

In [ ]:
potential_target_cols = ['fraud', 'is_fraud', 'label', 'target', 'class', 'fraudulent']
target_col = None

for col in sample_data.columns:
    if col.lower() in potential_target_cols or 'fraud' in col.lower():
        target_col = col
        break

if target_col:
    print(f"Found target column: {target_col}")
    print(f"Target distribution:")
    print(sample_data[target_col].value_counts())
    print(f"\nFraud rate: {sample_data[target_col].mean():.4f}")
else:
    print("No obvious fraud column found. Let's examine all columns:")
    for col in sample_data.columns:
        if sample_data[col].dtype in ['object', 'bool'] or sample_data[col].nunique() <= 10:
            print(f"\n{col} distribution:")
            print(sample_data[col].value_counts())

In [ ]:
def load_multiple_days(file_list, max_files=None):
    """Load multiple pickle files and combine into one dataframe"""
    if max_files:
        file_list = file_list[:max_files]
    
    all_data = []
    for i, file_name in enumerate(file_list):
        file_path = os.path.join(data_path, file_name)
        daily_data = load_daily_data(file_path)
        
        date_str = file_name.replace('.pkl', '')
        daily_data['date'] = date_str
        daily_data['day_of_week'] = pd.to_datetime(date_str).dayofweek
        daily_data['month'] = pd.to_datetime(date_str).month
        
        all_data.append(daily_data)
        
        if (i + 1) % 10 == 0:
            print(f"Loaded {i + 1} files...")
    
    combined_data = pd.concat(all_data, ignore_index=True)
    print(f"\nCombined dataset shape: {combined_data.shape}")
    return combined_data

print("Loading first 30 days of data for analysis...")
df_sample = load_multiple_days(pickle_files, max_files=30)

print(f"Sample dataset shape: {df_sample.shape}")
print(f"Date range: {df_sample['date'].min()} to {df_sample['date'].max()}")

In [ ]:
print("Combined Dataset Analysis:")
print(df_sample.info())

if target_col and target_col in df_sample.columns:
    print(f"\nTarget column '{target_col}' distribution:")
    print(df_sample[target_col].value_counts())
    print(f"Overall fraud rate: {df_sample[target_col].mean():.4f}")
    
    daily_fraud_rate = df_sample.groupby('date')[target_col].agg(['count', 'sum', 'mean'])
    daily_fraud_rate.columns = ['total_transactions', 'fraud_count', 'fraud_rate']
    
    print(f"\nDaily fraud statistics:")
    print(daily_fraud_rate.describe())
else:
    print("\nExamining potential target columns:")
    for col in df_sample.columns:
        if df_sample[col].dtype in ['int64', 'float64'] and df_sample[col].nunique() == 2:
            print(f"\nBinary column '{col}':")
            print(df_sample[col].value_counts())
            print(f"Rate: {df_sample[col].mean():.4f}")

In [ ]:
def plot_data_overview(df, target_col=None):
    """Create overview plots of the transaction data"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    daily_volume = df.groupby('date').size()
    axes[0, 0].plot(range(len(daily_volume)), daily_volume.values)
    axes[0, 0].set_title('Daily Transaction Volume')
    axes[0, 0].set_xlabel('Days')
    axes[0, 0].set_ylabel('Number of Transactions')
    axes[0, 0].grid(True)
    
    dow_volume = df.groupby('day_of_week').size()
    dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    axes[0, 1].bar(range(7), dow_volume.values)
    axes[0, 1].set_title('Transactions by Day of Week')
    axes[0, 1].set_xlabel('Day of Week')
    axes[0, 1].set_ylabel('Number of Transactions')
    axes[0, 1].set_xticks(range(7))
    axes[0, 1].set_xticklabels(dow_names)
    
    monthly_volume = df.groupby('month').size()
    axes[0, 2].bar(monthly_volume.index, monthly_volume.values)
    axes[0, 2].set_title('Transactions by Month')
    axes[0, 2].set_xlabel('Month')
    axes[0, 2].set_ylabel('Number of Transactions')
    
    if target_col and target_col in df.columns:
        daily_fraud = df.groupby('date')[target_col].mean()
        axes[1, 0].plot(range(len(daily_fraud)), daily_fraud.values, color='red')
        axes[1, 0].set_title('Daily Fraud Rate')
        axes[1, 0].set_xlabel('Days')
        axes[1, 0].set_ylabel('Fraud Rate')
        axes[1, 0].grid(True)
        
        dow_fraud = df.groupby('day_of_week')[target_col].mean()
        axes[1, 1].bar(range(7), dow_fraud.values, color='red', alpha=0.7)
        axes[1, 1].set_title('Fraud Rate by Day of Week')
        axes[1, 1].set_xlabel('Day of Week')
        axes[1, 1].set_ylabel('Fraud Rate')
        axes[1, 1].set_xticks(range(7))
        axes[1, 1].set_xticklabels(dow_names)
        
        monthly_fraud = df.groupby('month')[target_col].mean()
        axes[1, 2].bar(monthly_fraud.index, monthly_fraud.values, color='red', alpha=0.7)
        axes[1, 2].set_title('Fraud Rate by Month')
        axes[1, 2].set_xlabel('Month')
        axes[1, 2].set_ylabel('Fraud Rate')
    else:
        numeric_cols = df.select_dtypes(include=[np.number]).columns[:3]
        for i, col in enumerate(numeric_cols):
            if i < 3:
                axes[1, i].hist(df[col].dropna(), bins=50, alpha=0.7)
                axes[1, i].set_title(f'Distribution of {col}')
                axes[1, i].set_xlabel(col)
                axes[1, i].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()

plot_data_overview(df_sample, target_col)

In [ ]:
def create_features(df, target_col=None):
    """Create features for fraud detection"""
    df_features = df.copy()
    
    exclude_cols = ['date']
    if target_col:
        exclude_cols.append(target_col)
    
    feature_cols = [col for col in df_features.columns if col not in exclude_cols]
    
    categorical_cols = df_features[feature_cols].select_dtypes(include=['object']).columns
    
    print(f"Categorical columns found: {list(categorical_cols)}")
    
    le = LabelEncoder()
    for col in categorical_cols:
        df_features[col + '_encoded'] = le.fit_transform(df_features[col].astype(str))
        feature_cols.append(col + '_encoded')
        feature_cols.remove(col)
    
    df_features[feature_cols] = df_features[feature_cols].fillna(df_features[feature_cols].median())
    
    df_features['is_weekend'] = df_features['day_of_week'].isin([5, 6]).astype(int)
    
    if 'is_weekend' not in feature_cols:
        feature_cols.append('is_weekend')
    
    print(f"Total features created: {len(feature_cols)}")
    print(f"Feature columns: {feature_cols}")
    
    return df_features, feature_cols

df_processed, feature_columns = create_features(df_sample, target_col)

print(f"\nProcessed dataset shape: {df_processed.shape}")
print(f"Number of features: {len(feature_columns)}")

In [ ]:
if target_col and target_col in df_processed.columns:
    print("Preparing data for modeling...")
    
    X = df_processed[feature_columns]
    y = df_processed[target_col]
    
    print(f"Feature matrix shape: {X.shape}")
    print(f"Target vector shape: {y.shape}")
    print(f"Fraud rate: {y.mean():.4f}")
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"\nTraining set: {X_train.shape[0]} samples")
    print(f"Test set: {X_test.shape[0]} samples")
    print(f"Training fraud rate: {y_train.mean():.4f}")
    print(f"Test fraud rate: {y_test.mean():.4f}")
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    print("\nData preprocessing completed!")

else:
    print("No target column found. This appears to be an unsupervised fraud detection problem.")
    print("We'll use anomaly detection techniques.")
    
    X = df_processed[feature_columns]
    print(f"Feature matrix shape: {X.shape}")
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    print("Data prepared for anomaly detection!")

In [ ]:
if target_col and target_col in df_processed.columns:
    print("Training multiple models for fraud detection...")
    
    models = {
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
        'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='logloss'),
        'Gradient Boosting': GradientBoostingClassifier(random_state=42),
        'Decision Tree': DecisionTreeClassifier(random_state=42),
        'SVM': SVC(random_state=42, probability=True),
        'Naive Bayes': GaussianNB(),
        'KNN': KNeighborsClassifier()
    }
 
    results = {}
    
    for name, model in models.items():
        print(f"\nTraining {name}...")
        
        model.fit(X_train_scaled, y_train)
        
        y_pred = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else y_pred
        
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        try:
            auc_roc = roc_auc_score(y_test, y_pred_proba)
        except:
            auc_roc = roc_auc_score(y_test, y_pred)
        
        results[name] = {
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1,
            'AUC-ROC': auc_roc
        }
        
        print(f"Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, AUC: {auc_roc:.4f}")
    
    results_df = pd.DataFrame(results).T
    results_df = results_df.round(4)
    
    print("\n" + "="*80)
    print("MODEL COMPARISON RESULTS:")
    print("="*80)
    print(results_df)
    
    best_model_name = results_df['F1-Score'].idxmax()
    print(f"\nBest model by F1-Score: {best_model_name}")
    print(f"Best F1-Score: {results_df.loc[best_model_name, 'F1-Score']:.4f}")

else:
    print("Implementing anomaly detection for fraud detection...")
    
    iso_forest = IsolationForest(contamination=0.1, random_state=42)
    anomaly_labels = iso_forest.fit_predict(X_scaled)
    
    anomaly_binary = (anomaly_labels == -1).astype(int)
    
    print(f"Anomalies detected: {anomaly_binary.sum()}")
    print(f"Anomaly rate: {anomaly_binary.mean():.4f}")
    
    df_processed['anomaly_score'] = iso_forest.decision_function(X_scaled)
    df_processed['is_anomaly'] = anomaly_binary
    
    print("Anomaly detection completed!")

In [ ]:
if target_col and target_col in df_processed.columns:
    print(f"Detailed evaluation of {best_model_name}:")
    
    best_model = models[best_model_name]
    
    y_pred_best = best_model.predict(X_test_scaled)
    y_pred_proba_best = best_model.predict_proba(X_test_scaled)[:, 1] if hasattr(best_model, 'predict_proba') else y_pred_best
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_best))
    
    cm = confusion_matrix(y_test, y_pred_best)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1)
    ax1.set_title(f'Confusion Matrix - {best_model_name}')
    ax1.set_xlabel('Predicted')
    ax1.set_ylabel('Actual')
    
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba_best)
    auc_score = roc_auc_score(y_test, y_pred_proba_best)
    
    ax2.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc_score:.3f})')
    ax2.plot([0, 1], [0, 1], 'k--', label='Random')
    ax2.set_xlabel('False Positive Rate')
    ax2.set_ylabel('True Positive Rate')
    ax2.set_title(f'ROC Curve - {best_model_name}')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()

else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    axes[0].hist(df_processed['anomaly_score'], bins=50, alpha=0.7)
    axes[0].set_title('Anomaly Score Distribution')
    axes[0].set_xlabel('Anomaly Score')
    axes[0].set_ylabel('Frequency')
    axes[0].axvline(x=0, color='red', linestyle='--', label='Threshold')
    axes[0].legend()
    daily_anomalies = df_processed.groupby('date')['is_anomaly'].mean()
    axes[1].plot(range(len(daily_anomalies)), daily_anomalies.values)
    axes[1].set_title('Daily Anomaly Rate')
    axes[1].set_xlabel('Days')
    axes[1].set_ylabel('Anomaly Rate')
    axes[1].grid(True)
    
    dow_anomalies = df_processed.groupby('day_of_week')['is_anomaly'].mean()
    dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    axes[2].bar(range(7), dow_anomalies.values, alpha=0.7)
    axes[2].set_title('Anomaly Rate by Day of Week')
    axes[2].set_xlabel('Day of Week')
    axes[2].set_ylabel('Anomaly Rate')
    axes[2].set_xticks(range(7))
    axes[2].set_xticklabels(dow_names)
    
    plt.tight_layout()
    plt.show()

In [ ]:
if target_col and target_col in df_processed.columns:
    if 'Random Forest' in models:
        rf_model = models['Random Forest']
        feature_importance = pd.DataFrame({
            'feature': feature_columns,
            'importance': rf_model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        print("Top 15 Most Important Features (Random Forest):")
        print(feature_importance.head(15))
        
        plt.figure(figsize=(12, 8))
        top_features = feature_importance.head(15)
        plt.barh(range(len(top_features)), top_features['importance'])
        plt.yticks(range(len(top_features)), top_features['feature'])
        plt.xlabel('Feature Importance')
        plt.title('Top 15 Feature Importances (Random Forest)')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()

else:
    print("Feature importance analysis for anomaly detection:")
    print("Most anomalous transactions characteristics:")
    
    anomalous_transactions = df_processed[df_processed['is_anomaly'] == 1]
    normal_transactions = df_processed[df_processed['is_anomaly'] == 0]
    
    print(f"\nAnomalous transactions: {len(anomalous_transactions)}")
    print(f"Normal transactions: {len(normal_transactions)}")
    
    numeric_features = df_processed[feature_columns].select_dtypes(include=[np.number]).columns
    comparison = pd.DataFrame({
        'Normal_Mean': normal_transactions[numeric_features].mean(),
        'Anomalous_Mean': anomalous_transactions[numeric_features].mean()
    })
    comparison['Difference'] = comparison['Anomalous_Mean'] - comparison['Normal_Mean']
    comparison = comparison.sort_values('Difference', key=abs, ascending=False)
    
    print("\nTop 10 features with largest differences between normal and anomalous:")
    print(comparison.head(10))

In [ ]:
def predict_fraud_transaction(transaction_data, model=None, scaler=None, feature_cols=None, threshold=0.5):
    """
    Predict if a single transaction is fraudulent
    
    Parameters:
    transaction_data: dict or Series containing transaction features
    model: trained model
    scaler: fitted StandardScaler
    feature_cols: list of feature column names
    threshold: prediction threshold
    
    Returns:
    dict with prediction results
    """
    if isinstance(transaction_data, dict):
        transaction_df = pd.DataFrame([transaction_data])
    else:
        transaction_df = pd.DataFrame([transaction_data])
    
    transaction_features = transaction_df[feature_cols]
    transaction_scaled = scaler.transform(transaction_features)
    
    if hasattr(model, 'predict_proba'):
        fraud_probability = model.predict_proba(transaction_scaled)[0, 1]
        is_fraud = fraud_probability > threshold
    else:
        is_fraud = model.predict(transaction_scaled)[0]
        fraud_probability = float(is_fraud)
    
    return {
        'is_fraud': bool(is_fraud),
        'fraud_probability': fraud_probability,
        'risk_level': 'High' if fraud_probability > 0.7 else 'Medium' if fraud_probability > 0.3 else 'Low'
    }

if target_col and target_col in df_processed.columns:
    print("Example fraud prediction:")
    
    sample_transaction = X_test.iloc[0]
    
    prediction = predict_fraud_transaction(
        sample_transaction, 
        model=models[best_model_name], 
        scaler=scaler, 
        feature_cols=feature_columns
    )
    
    print(f"Sample transaction prediction:")
    print(f"Is Fraud: {prediction['is_fraud']}")
    print(f"Fraud Probability: {prediction['fraud_probability']:.4f}")
    print(f"Risk Level: {prediction['risk_level']}")
    print(f"Actual label: {y_test.iloc[0]}")

else:
    print("Anomaly detection deployment example:")
    
    def detect_anomaly_transaction(transaction_data, model, scaler, feature_cols):
        if isinstance(transaction_data, dict):
            transaction_df = pd.DataFrame([transaction_data])
        else:
            transaction_df = pd.DataFrame([transaction_data])
        
        transaction_features = transaction_df[feature_cols]
        transaction_scaled = scaler.transform(transaction_features)
        
        anomaly_score = model.decision_function(transaction_scaled)[0]
        is_anomaly = model.predict(transaction_scaled)[0] == -1
        
        return {
            'is_anomaly': bool(is_anomaly),
            'anomaly_score': anomaly_score,
            'risk_level': 'High' if anomaly_score < -0.2 else 'Medium' if anomaly_score < 0 else 'Low'
        }
    
    sample_transaction = X.iloc[0]
    anomaly_prediction = detect_anomaly_transaction(
        sample_transaction, 
        iso_forest, 
        scaler, 
        feature_columns
    )
    
    print(f"Sample transaction anomaly detection:")
    print(f"Is Anomaly: {anomaly_prediction['is_anomaly']}")
    print(f"Anomaly Score: {anomaly_prediction['anomaly_score']:.4f}")
    print(f"Risk Level: {anomaly_prediction['risk_level']}")

In [ ]:
import joblib

if target_col and target_col in df_processed.columns:
    joblib.dump(models[best_model_name], 'fraud_detection_model.pkl')
    joblib.dump(scaler, 'fraud_detection_scaler.pkl')
    
    metadata = {
        'feature_columns': feature_columns,
        'target_column': target_col,
        'best_model': best_model_name,
        'model_performance': results_df.loc[best_model_name].to_dict()
    }
    
    with open('fraud_detection_metadata.pkl', 'wb') as f:
        pickle.dump(metadata, f)
    
    print(f"Model saved: fraud_detection_model.pkl")
    print(f"Scaler saved: fraud_detection_scaler.pkl")
    print(f"Metadata saved: fraud_detection_metadata.pkl")

else:
    joblib.dump(iso_forest, 'anomaly_detection_model.pkl')
    joblib.dump(scaler, 'anomaly_detection_scaler.pkl')
    
    metadata = {
        'feature_columns': feature_columns,
        'model_type': 'anomaly_detection',
        'contamination_rate': 0.1
    }
    
    with open('anomaly_detection_metadata.pkl', 'wb') as f:
        pickle.dump(metadata, f)
    
    print(f"Anomaly detection model saved: anomaly_detection_model.pkl")
    print(f"Scaler saved: anomaly_detection_scaler.pkl")
    print(f"Metadata saved: anomaly_detection_metadata.pkl")

print("\nAll models and preprocessing objects saved successfully!")